# Sesión 7 · Redes neuronales
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

**Problema:** en la sesión 6 clasificamos el voltaje del panel solar de la estación de Anapoima como ALTO o BAJO. Ahora vamos a predecir **el valor real del voltaje en voltios** (esto es *regresión*) usando una **red neuronal**, y a comprobar si realmente supera a una regresión lineal simple.

**Ruta:** datos → qué es una neurona → dividir y escalar → cómo aprende una red → línea base → red neuronal y sus hiperparámetros → curva de pérdida → efecto de los hiperparámetros → qué variables importan → Keras.

> Ejecute cada celda con `Shift + Enter`. Las celdas de PRÁCTICA son para completar.
> Antes de empezar, juegue 5 minutos en [TensorFlow Playground](https://playground.tensorflow.org) para ver una red aprendiendo.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
print('Librerías cargadas')

## PARTE 1 · Los datos
Usamos el mismo histórico de la sesión 6 (estación de Anapoima, publicada en GitHub) y la misma limpieza mínima:
1. Una lectura por `fecha_hora` (se eliminan repetidas).
2. Solo de día (`irradiancia_wm2 > 50`): de noche el voltaje es ~0 y predecirlo no tiene gracia.

Además **ordenamos por fecha**, porque en la PARTE 4 dividiremos el tiempo en pasado (entrenamiento) y futuro (prueba).

In [ ]:
url = 'https://raw.githubusercontent.com/anclopezg/curso-analisis-datos-energias-renovables/main/docs/datasets/datos_anapoima.csv'
df = pd.read_csv(url, parse_dates=['fecha_hora'])
df = df.drop_duplicates(subset='fecha_hora').sort_values('fecha_hora')
df = df[df['irradiancia_wm2'] > 50]

features = ['temperatura_c', 'irradiancia_wm2', 'velocidad_viento_kmh', 'humedad_relativa', 'presion_hpa']
df = df.dropna(subset=features + ['ws_panel_volt']).reset_index(drop=True)
print(f'{len(df)} lecturas de día, del {df["fecha_hora"].min()} al {df["fecha_hora"].max()}')
df[features + ['ws_panel_volt']].describe().round(2)

### PRÁCTICA 1


1. ¿Cuál es el voltaje mínimo, el máximo y el promedio?
2. ¿Cuántas lecturas hay por mes? (*Pista:* `df['fecha_hora'].dt.to_period('M').value_counts().sort_index()`)
3. ¿Hay meses con muy pocos datos? ¿Por qué podría importar para dividir en entrenamiento y prueba?

In [ ]:
print(df['ws_panel_volt'].agg(['min', 'max', 'mean']).round(2))
print(df['fecha_hora'].dt.to_period('M').value_counts().sort_index())
# Si el último tramo (el de prueba) tiene otra época del año, el modelo se evalúa en condiciones algo distintas a las que aprendió.

## PARTE 2 · Regresión: predecir un número
En la sesión 6 convertimos el voltaje en una categoría con un umbral. Ahora **no lo convertimos**: la red debe devolver el número (por ejemplo 12.8 V). Eso se llama **regresión**.

Una consecuencia: ya no usamos accuracy ni matriz de confusión. Medimos **cuánto se equivoca en voltios**:
- **MAE** (error absoluto medio): el error promedio, en voltios. Fácil de interpretar: MAE = 0.35 significa "en promedio se equivoca por 0.35 V".
- **RMSE:** como el MAE, pero castiga más los errores grandes.
- **R²:** qué fracción de la variación del voltaje explica el modelo (1 = perfecto, 0 = no mejor que predecir siempre el promedio).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
df['ws_panel_volt'].plot(kind='hist', bins=50, edgecolor='black', ax=ax[0])
ax[0].set_title('Distribución del voltaje (de día)'); ax[0].set_xlabel('Voltaje (V)')
ax[1].scatter(df['irradiancia_wm2'], df['ws_panel_volt'], s=3, alpha=0.3)
ax[1].set_title('Irradiancia vs voltaje'); ax[1].set_xlabel('Irradiancia (W/m2)'); ax[1].set_ylabel('Voltaje (V)')
plt.tight_layout(); plt.show()

## PARTE 3 · ¿Qué es una neurona?
Una neurona artificial hace **tres cosas**:
1. **Multiplica** cada entrada por un peso: `x1·w1, x2·w2, ...` (el peso dice cuánta importancia le da a esa entrada).
2. **Suma** todo y añade un sesgo (*bias*): `z = w1·x1 + w2·x2 + ... + b`.
3. Pasa el resultado por una **función de activación** `f`: `salida = f(z)`.

En fórmula: **`a = f( Σ wᵢ·xᵢ + b )`**

La función de activación es lo que permite que la red aprenda relaciones **curvas**. Sin ella, por muchas capas que tenga, la red sería equivalente a una regresión lineal.

| Activación | Fórmula | Qué hace |
|---|---|---|
| **ReLU** | `max(0, z)` | Deja pasar los positivos y apaga los negativos. La más usada: simple y rápida |
| **tanh** | `(eᶻ − e⁻ᶻ)/(eᶻ + e⁻ᶻ)` | Aplasta el valor entre −1 y 1 |
| **sigmoide (logistic)** | `1/(1 + e⁻ᶻ)` | Aplasta el valor entre 0 y 1 |
| **identity** | `z` | No transforma nada: la red queda lineal |

In [ ]:
# Una neurona "a mano" con una lectura ya escalada (5 entradas: temp, irradiancia, viento, humedad, presión)
x = np.array([0.5, 1.2, -0.3, -0.8, 0.1])
w = np.array([0.2, 0.8, -0.1, -0.3, 0.05])    # un peso por entrada
b = 0.1                                       # sesgo

z = np.dot(w, x) + b                          # suma ponderada
a = max(0, z)                                 # activación ReLU
print(f'z = {z:.3f}  ->  salida (ReLU) = {a:.3f}')

# Las cuatro activaciones
zs = np.linspace(-4, 4, 200)
fig, ax = plt.subplots(1, 4, figsize=(14, 2.8), sharey=False)
for axi, (nombre, f) in zip(ax, [('ReLU', lambda z: np.maximum(0, z)), ('tanh', np.tanh),
                                  ('sigmoide', lambda z: 1/(1+np.exp(-z))), ('identity', lambda z: z)]):
    axi.plot(zs, f(zs)); axi.set_title(nombre); axi.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### PRÁCTICA 2


1. Calcule la salida de la misma neurona con activación **tanh** y con **sigmoide**. ¿En qué rango queda cada una?
2. Cambie el peso de la irradiancia (`w[1]`) de 0.8 a −0.8. ¿Cómo cambia `z`? ¿Qué significa un peso negativo?

*Pista:* `np.tanh(z)` y `1/(1 + np.exp(-z))`.

In [ ]:
print('tanh    :', round(np.tanh(z), 3), '(entre -1 y 1)')
print('sigmoide:', round(1/(1 + np.exp(-z)), 3), '(entre 0 y 1)')
w2 = w.copy(); w2[1] = -0.8
z2 = np.dot(w2, x) + b
print(f'z con w[1]=0.8: {z:.3f} | con w[1]=-0.8: {z2:.3f}')
# Un peso negativo hace que una entrada alta REDUZCA la salida de la neurona.

## PARTE 4 · Dividir y escalar
**División cronológica.** Como el objetivo es predecir lo que viene, entrenamos con el **primer 80 % del tiempo** (pasado) y probamos con el **último 20 %** (futuro). Es una prueba más honesta que mezclar filas al azar, porque lecturas vecinas en el tiempo se parecen mucho y mezclarlas haría que el modelo "viera" casi lo mismo que va a evaluar (lo comprobaremos en la PRÁCTICA 8).

**Escalado.** Las redes aprenden con gradientes y son muy sensibles a la escala: si la presión vale ~930 y el viento ~5, la presión dominaría. Escalamos cada variable a media 0 y desviación 1 (`StandardScaler`). También escalamos el **voltaje** (`y`), y luego revertimos el escalado para ver los errores en voltios. Los escaladores se ajustan (`fit`) **solo con entrenamiento**.

In [ ]:
corte = int(len(df) * 0.8)
train, test = df.iloc[:corte], df.iloc[corte:]
print(f'Entrenamiento: {len(train)} lecturas (hasta {train["fecha_hora"].max():%Y-%m-%d})')
print(f'Prueba:        {len(test)} lecturas (desde {test["fecha_hora"].min():%Y-%m-%d})')

esc_x = StandardScaler().fit(train[features])
esc_y = StandardScaler().fit(train[['ws_panel_volt']])

X_train_s = esc_x.transform(train[features])
X_test_s = esc_x.transform(test[features])
y_train_s = esc_y.transform(train[['ws_panel_volt']]).ravel()
y_test_s = esc_y.transform(test[['ws_panel_volt']]).ravel()
y_test = test['ws_panel_volt'].values          # voltaje real en voltios


def a_voltios(pred_escalada):
    return esc_y.inverse_transform(np.asarray(pred_escalada).reshape(-1, 1)).ravel()


def evaluar(nombre, pred_voltios):
    return {'Modelo': nombre,
            'MAE (V)': round(mean_absolute_error(y_test, pred_voltios), 3),
            'RMSE (V)': round(np.sqrt(mean_squared_error(y_test, pred_voltios)), 3),
            'R2': round(r2_score(y_test, pred_voltios), 3)}

## PARTE 5 · ¿Cómo aprende una red?
Entrenar es **ajustar los pesos** para que el error baje. El ciclo es:
1. **Predecir** con los pesos actuales.
2. **Medir el error** con una *función de pérdida* (aquí el error cuadrático medio, MSE).
3. **Calcular el gradiente**: hacia dónde hay que mover cada peso para que el error baje.
4. **Actualizar** los pesos dando un paso pequeño: `w ← w − tasa_de_aprendizaje · gradiente`.
5. Repetir. Una pasada por todos los datos de entrenamiento es una **época**.

La **tasa de aprendizaje** (*learning rate*) es el tamaño del paso. Lo vemos con el modelo más simple posible: un solo peso, `predicción = w · irradiancia`. Veremos el mismo ciclo con tres tamaños de paso.

In [ ]:
x_irr = X_train_s[:, 1]        # irradiancia escalada
y_obj = y_train_s


def descenso(lr, pasos=30):
    w = -1.0                                    # peso inicial, lejos de la solución
    perdidas = []
    for _ in range(pasos):
        error = w * x_irr - y_obj
        perdidas.append(np.mean(error ** 2))    # MSE
        gradiente = 2 * np.mean(error * x_irr)  # derivada del MSE respecto a w
        w = w - lr * gradiente                  # paso de descenso
    return perdidas, w


plt.figure(figsize=(8, 4))
for lr, etiqueta in [(0.01, 'muy pequeña (0.01)'), (0.4, 'adecuada (0.4)'), (1.1, 'demasiado grande (1.1)')]:
    perdidas, w_final = descenso(lr)
    plt.plot(perdidas, label=f'{etiqueta}: w final = {w_final:.2f}')
plt.yscale('log'); plt.xlabel('paso'); plt.ylabel('pérdida (MSE, escala log)')
plt.title('Efecto de la tasa de aprendizaje'); plt.legend(); plt.grid(alpha=0.3); plt.show()

- **Muy pequeña:** la pérdida baja pero tan despacio que no llega en 30 pasos.
- **Adecuada:** converge rápido.
- **Demasiado grande:** cada paso se pasa de largo y el error **crece** en lugar de bajar (la red "explota").

Las redes reales hacen esto con miles de pesos a la vez, y usan variantes como **Adam**, que ajusta el tamaño del paso de cada peso automáticamente.

### PRÁCTICA 3


Use la función `descenso` con `lr = 0.9`, `lr = 1.0` y `lr = 2.0`. Imprima la pérdida final de cada uno (`perdidas[-1]`).
¿Cuál converge, cuál se queda oscilando y cuál diverge? (*Pista:* como las variables están escaladas, el límite está cerca de `lr = 1`.)

In [ ]:
for lr in [0.9, 1.0, 2.0]:
    perdidas, w_final = descenso(lr)
    print(f'lr={lr}: pérdida final = {perdidas[-1]:.4g}, w final = {w_final:.2f}')
# 0.9 oscila pero converge; 1.0 queda dando saltos sin bajar; 2.0 diverge (la pérdida explota).

## PARTE 6 · Línea base: regresión lineal
Antes de usar una red neuronal hay que tener un **punto de comparación sencillo**. Si la red no mejora a una regresión lineal, no vale la pena su complejidad. La regresión lineal ajusta un único plano: `voltaje = w1·temp + w2·irradiancia + ... + b`.

In [ ]:
lineal = LinearRegression().fit(X_train_s, y_train_s)
pred_lineal = a_voltios(lineal.predict(X_test_s))
res_lineal = evaluar('Regresión lineal', pred_lineal)
res_lineal

### PRÁCTICA 4


1. Calcule el MAE de un "modelo tonto" que siempre predice el **promedio del voltaje de entrenamiento**. (*Pista:* `train['ws_panel_volt'].mean()` y un arreglo de ese valor con `np.full(len(y_test), ...)`.)
2. ¿Cuánto mejora la regresión lineal respecto a ese modelo tonto?

In [ ]:
tonto = np.full(len(y_test), train['ws_panel_volt'].mean())
print('Modelo tonto:', evaluar('Promedio', tonto))
print('Lineal      :', res_lineal)
# Cualquier modelo útil debe superar claramente al promedio; y una red debe superar a la lineal para justificar su complejidad.

## PARTE 7 · La red neuronal (MLPRegressor)
Una **red neuronal "perceptrón multicapa"** (MLP) es un conjunto de neuronas organizadas en capas:
- **Capa de entrada:** las 5 variables del clima (no calcula nada).
- **Capas ocultas:** cada neurona recibe las salidas de la capa anterior y aplica `f(Σ w·x + b)`. Aquí 2 capas ocultas, de 32 y 16 neuronas.
- **Capa de salida:** 1 neurona que entrega el voltaje (escalado).

Arquitectura `5 → 32 → 16 → 1`. Número de parámetros (pesos + sesgos) a ajustar: `(5·32 + 32) + (32·16 + 16) + (16·1 + 1) = 737`.

### Hiperparámetros principales (`MLPRegressor`)
| Hiperparámetro | Qué controla | Si lo aumentas / cambias |
|---|---|---|
| `hidden_layer_sizes` (def. `(100,)`) | Capas ocultas y neuronas por capa | Más neuronas/capas → más capacidad para captar relaciones complejas, pero más lento y con riesgo de sobreajuste. Muy pocas (por ejemplo `(2,)`) → la red queda corta (subajuste) |
| `activation` (`'relu'`, `'tanh'`, `'logistic'`, `'identity'`) | Función de cada neurona | `relu` es el estándar; `tanh` suele dar resultados parecidos; `identity` vuelve la red lineal |
| `solver` (`'adam'`, `'sgd'`, `'lbfgs'`) | Algoritmo que actualiza los pesos | `adam` funciona bien casi siempre; `sgd` es más sensible a la tasa de aprendizaje; `lbfgs` es bueno con pocos datos |
| `learning_rate_init` (def. 0.001) | Tamaño del paso | Muy grande → la red no converge (el error se dispara); muy pequeña → aprende lentísimo y puede cortarse antes de terminar |
| `alpha` (def. 0.0001) | Penalización L2 a los pesos grandes (regularización) | Más alto → modelo más suave y menos sobreajustado, pero si se pasa, queda corto |
| `batch_size` (def. `'auto'` = min(200, n)) | Cuántas filas se usan por cada actualización de pesos | Pequeño → actualizaciones más ruidosas pero frecuentes; grande → más estables pero menos frecuentes |
| `max_iter` (def. 200) | Máximo de épocas | Muy bajo → entrenamiento incompleto (aviso de "no convergió") |
| `early_stopping` + `validation_fraction` + `n_iter_no_change` | Parar cuando la validación deja de mejorar | Evita sobreajustar y ahorra tiempo: aparta un % de entrenamiento como validación y detiene el entrenamiento si no mejora durante `n_iter_no_change` épocas |
| `random_state` | Semilla de los pesos iniciales | No mejora el modelo; solo hace el resultado **reproducible** (con otra semilla el resultado cambia un poco) |

In [ ]:
red = MLPRegressor(hidden_layer_sizes=(32, 16), activation='relu', solver='adam',
                   learning_rate_init=0.001, alpha=0.0001, max_iter=300,
                   early_stopping=True, random_state=0)
red.fit(X_train_s, y_train_s)
pred_red = a_voltios(red.predict(X_test_s))

print(pd.DataFrame([res_lineal, evaluar('Red neuronal (32,16)', pred_red)]).set_index('Modelo'))
print(f'\nÉpocas realizadas: {red.n_iter_}')
print('Forma de las matrices de pesos:', [w.shape for w in red.coefs_])
print('Total de parámetros:', sum(w.size for w in red.coefs_) + sum(b.size for b in red.intercepts_))

## PARTE 8 · Curva de pérdida
Guardamos la pérdida en cada época. Es el **termómetro del entrenamiento**:
- Si la pérdida de entrenamiento **baja y se estabiliza**: la red está aprendiendo y converge.
- Si el puntaje en validación deja de mejorar (o empeora) mientras el de entrenamiento sigue mejorando: **sobreajuste**. `early_stopping` corta ahí.
- Si la pérdida **oscila o sube**: tasa de aprendizaje muy grande.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(red.loss_curve_); ax[0].set_title('Pérdida de entrenamiento'); ax[0].set_xlabel('época'); ax[0].set_ylabel('pérdida')
ax[1].plot(red.validation_scores_, color='tab:orange'); ax[1].set_title('Puntaje en validación (R2)'); ax[1].set_xlabel('época')
for a_ in ax: a_.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## PARTE 9 · Real contra predicho
Dos formas de ver si predice bien:
- **Dispersión real vs predicho:** si el modelo fuera perfecto, todos los puntos caerían sobre la diagonal.
- **Serie en el tiempo:** una semana de lecturas reales contra lo que predijo cada modelo.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].scatter(y_test, pred_red, s=3, alpha=0.3)
lim = [y_test.min(), y_test.max()]
ax[0].plot(lim, lim, 'r--', label='predicción perfecta')
ax[0].set_xlabel('Voltaje real (V)'); ax[0].set_ylabel('Voltaje predicho (V)'); ax[0].set_title('Red neuronal'); ax[0].legend()

n = 150
ax[1].plot(y_test[:n], label='real'); ax[1].plot(pred_red[:n], label='red'); ax[1].plot(pred_lineal[:n], label='lineal', alpha=0.6)
ax[1].set_title(f'Primeras {n} lecturas de prueba'); ax[1].set_xlabel('lectura'); ax[1].set_ylabel('Voltaje (V)'); ax[1].legend()
plt.tight_layout(); plt.show()

## PARTE 10 · Efecto de los hiperparámetros
Aquí se ve en la práctica lo que dice la tabla de la PARTE 7. Para comparar con justicia, entrenamos varias redes cambiando **un solo hiperparámetro a la vez**. Empezamos con la **arquitectura**: pocas neuronas vs. muchas.

In [ ]:
def entrenar(**kw):
    params = dict(hidden_layer_sizes=(32, 16), max_iter=300, early_stopping=True, random_state=0)
    params.update(kw)
    m = MLPRegressor(**params).fit(X_train_s, y_train_s)
    return m, a_voltios(m.predict(X_test_s))


filas = []
for arq in [(2,), (8,), (32, 16), (128, 64)]:
    m, p = entrenar(hidden_layer_sizes=arq)
    fila = evaluar(f'arquitectura {arq}', p); fila['épocas'] = m.n_iter_; filas.append(fila)
print(pd.DataFrame(filas).set_index('Modelo'))
print('\nReferencia, regresión lineal:', res_lineal['MAE (V)'], 'V de MAE')

Con `(2,)` la red casi no mejora a la regresión lineal: **no tiene capacidad suficiente**. Al pasar a `(32, 16)` el error baja claramente. Pasar a `(128, 64)` (muchas más neuronas) casi no mejora: hay un punto de rendimientos decrecientes en el que más capacidad solo cuesta más tiempo.

### PRÁCTICA 5


Entrene redes con `learning_rate_init` = 0.00001, 0.001 y 1.0 (use la función `entrenar(learning_rate_init=...)`). Para cada una muestre MAE y épocas.
1. ¿Cuál es demasiado lenta?
2. ¿Cuál no logra aprender? ¿Cómo lo reconoce en el MAE y en R²?

In [ ]:
filas = []
for lr in [0.00001, 0.001, 1.0]:
    m, p = entrenar(learning_rate_init=lr)
    fila = evaluar(f'learning_rate_init={lr}', p); fila['épocas'] = m.n_iter_; filas.append(fila)
print(pd.DataFrame(filas).set_index('Modelo'))
# 1e-5: aprende, pero muy despacio (necesita muchas más épocas y queda peor). 1.0: pasos enormes, no converge: R2 ~ 0 (no mejor que predecir el promedio).

### PRÁCTICA 6


Pruebe la **regularización** y la **activación**:
1. `alpha` = 0.0001, 0.1 y 1.0.
2. `activation` = `'relu'`, `'tanh'` e `'identity'`.

¿Qué pasa con `'identity'`? ¿A qué modelo ya conocido se parece su resultado y por qué?

In [ ]:
filas = []
for alpha in [0.0001, 0.1, 1.0]:
    m, p = entrenar(alpha=alpha); filas.append(evaluar(f'alpha={alpha}', p))
for act in ['relu', 'tanh', 'identity']:
    m, p = entrenar(activation=act); filas.append(evaluar(f'activation={act}', p))
print(pd.DataFrame(filas).set_index('Modelo'))
# identity deja la red lineal: su resultado se parece al de la regresión lineal porque, sin activación no lineal, varias capas colapsan en una sola operación lineal.

## PARTE 11 · ¿Y si mezclamos el tiempo?
En la PARTE 4 dijimos que dividir por fecha es más honesto. Lo verificamos: entrenamos la misma red con una división **al azar** (como en la sesión 6) y comparamos. Si el resultado sale mejor, no es porque el modelo sea mejor, sino porque la prueba es **más fácil**: cada lectura de prueba tiene "vecinas" casi idénticas en el entrenamiento.

### PRÁCTICA 7


1. Divida `df` al azar 80/20 con `train_test_split(..., random_state=42)`.
2. Escale, entrene la misma red `(32, 16)` y calcule el MAE.
3. Compárelo con el MAE de la división cronológica. ¿Cuál parece mejor y cuál es más realista para predecir el futuro?

*Pista:* ajuste un escalador nuevo solo con el entrenamiento aleatorio.

In [ ]:
from sklearn.model_selection import train_test_split
tr, te = train_test_split(df, test_size=0.2, random_state=42)
ex = StandardScaler().fit(tr[features]); ey = StandardScaler().fit(tr[['ws_panel_volt']])
m = MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=300, early_stopping=True, random_state=0)
m.fit(ex.transform(tr[features]), ey.transform(tr[['ws_panel_volt']]).ravel())
p = ey.inverse_transform(m.predict(ex.transform(te[features])).reshape(-1, 1)).ravel()
print('MAE con división al azar     :', round(mean_absolute_error(te['ws_panel_volt'], p), 3))
print('MAE con división cronológica :', round(mean_absolute_error(y_test, pred_red), 3))
# El MAE al azar sale menor, pero es optimista: no mide qué pasará con datos futuros.

## PARTE 12 · ¿Qué variables importan? (importancia por permutación)
Una red neuronal es una "caja negra": no tiene `feature_importances_` como el Random Forest. Una técnica general es la **importancia por permutación**:
1. Se mide el error del modelo en el conjunto de prueba.
2. Se **desordena al azar** una sola variable (se rompe su relación con el voltaje) y se vuelve a medir.
3. **Cuanto más empeora el error, más importante era esa variable.**

Funciona con cualquier modelo. Se repite varias veces (`n_repeats`) para obtener un promedio y su variación.

**Cuidado al interpretar:** mide cuánto *depende esta red* de la variable. No es lo mismo que preguntar "¿qué pasa si entreno sin ella?", y si dos variables están relacionadas entre sí, la importancia se reparte entre ellas. La PRÁCTICA 8 explora esa diferencia.

In [ ]:
perm = permutation_importance(red, X_test_s, y_test_s, scoring='neg_mean_absolute_error', n_repeats=10, random_state=0)
imp = pd.Series(perm.importances_mean, index=features).sort_values()
imp.plot(kind='barh', xerr=pd.Series(perm.importances_std, index=features)[imp.index], figsize=(8, 3.5), color='tab:green')
plt.xlabel('Aumento del error (MAE escalado) al desordenar la variable'); plt.title('Importancia por permutación'); plt.grid(alpha=0.3); plt.show()

### PRÁCTICA 8


1. ¿Qué variable es la más importante? ¿Coincide con la que vimos en la sesión 6?
2. ¿Hay alguna casi irrelevante? Entrene una red **sin** esa variable y compare el MAE. ¿Empeora mucho?

*Pista:* quite la variable, reescale, reentrene y evalúe. Repita con varias `random_state` y compare los promedios: una sola corrida puede engañar.

In [ ]:
print(imp.sort_values(ascending=False).round(3))
menos = imp.index[0]                      # la de menor importancia
fm = [f for f in features if f != menos]
ex2 = StandardScaler().fit(train[fm])


def mae_con(vars_, esc, semilla):
    m = MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=300, early_stopping=True, random_state=semilla)
    m.fit(esc.transform(train[vars_]), y_train_s)
    return mean_absolute_error(y_test, a_voltios(m.predict(esc.transform(test[vars_]))))


todas = [mae_con(features, esc_x, s) for s in range(5)]
sin = [mae_con(fm, ex2, s) for s in range(5)]
print(f'MAE con todas      : {np.mean(todas):.3f} (min {min(todas):.3f}, max {max(todas):.3f})')
print(f'MAE sin {menos}: {np.mean(sin):.3f} (min {min(sin):.3f}, max {max(sin):.3f})')
# Se repite con 5 semillas porque una sola puede engañar. Aunque la permutación le dé poca importancia,
# quitarla y reentrenar empeora: la presión aporta información que las otras variables no cubren del todo.

## PARTE 13 · La misma red en Keras (opcional)
**Keras** (dentro de TensorFlow) es la librería que se usa en la práctica profesional para redes más grandes. La arquitectura es la misma `5 → 32 → 16 → 1`; cambia cómo se escribe. Conceptos equivalentes:

| scikit-learn | Keras |
|---|---|
| `hidden_layer_sizes=(32, 16)` | `Dense(32)`, `Dense(16)` |
| `max_iter` | `epochs` |
| `learning_rate_init` | `optimizer=Adam(learning_rate=...)` |
| `early_stopping=True` | callback `EarlyStopping` |
| `batch_size` | `batch_size` (def. 32) |

**Dos detalles importantes:**
- Aquí separamos una validación **al azar** del entrenamiento. Si usáramos `validation_split=0.2`, Keras toma el **último 20 %** sin mezclar, que en una serie de tiempo es un periodo distinto del resto; la parada anticipada corta entonces el entrenamiento demasiado pronto (en nuestras pruebas, a las ~15 épocas) y la red queda peor.
- El resultado de Keras **no será idéntico** al de scikit-learn aunque la arquitectura sea la misma: cambian la inicialización, el tamaño de lote y cómo se elige la parada. Varía también entre corridas. Compare con cautela y repita con otra semilla.

En Google Colab, TensorFlow ya viene instalado; si su entorno no lo tiene, la celda lo avisa.

In [ ]:
try:
    import tensorflow as tf
    tf.random.set_seed(0)
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train_s.shape[1],)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1)])
    modelo.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    parar = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    idx = np.random.RandomState(0).permutation(len(X_train_s))          # validación al azar (10 %)
    n_val = int(0.1 * len(idx)); val, trn = idx[:n_val], idx[n_val:]
    hist = modelo.fit(X_train_s[trn], y_train_s[trn], epochs=300, batch_size=32,
                      validation_data=(X_train_s[val], y_train_s[val]), callbacks=[parar], verbose=0)
    print('Épocas realizadas:', len(hist.history['loss']))
    pred_keras = a_voltios(modelo.predict(X_test_s, verbose=0))
    print(pd.DataFrame([res_lineal, evaluar('Red sklearn (32,16)', pred_red), evaluar('Red Keras (32,16)', pred_keras)]).set_index('Modelo'))
    plt.plot(hist.history['loss'], label='entrenamiento'); plt.plot(hist.history['val_loss'], label='validación')
    plt.xlabel('época'); plt.ylabel('pérdida (MSE)'); plt.legend(); plt.grid(alpha=0.3); plt.show()
except ImportError:
    print('TensorFlow no está instalado en este entorno; use la red de scikit-learn de la PARTE 7.')

## Resumen
- **Regresión:** predecimos el voltaje en voltios; medimos con MAE, RMSE y R².
- **Neurona:** `f(Σ w·x + b)`. La función de activación aporta la no linealidad; sin ella la red es una regresión lineal.
- **Entrenar:** predecir → medir el error → calcular el gradiente → ajustar los pesos con un paso (`learning rate`) → repetir por épocas.
- **Siempre una línea base** (promedio y regresión lineal) antes de celebrar el resultado de la red.
- **Hiperparámetros clave:** `hidden_layer_sizes` (capacidad), `learning_rate_init` (tamaño del paso), `alpha` (regularización), `activation`, `max_iter`/`early_stopping` (cuándo parar).
- **Curva de pérdida** y comparación train/validación: así se detecta subajuste, sobreajuste o un paso mal elegido.
- Dividir **por tiempo** da una evaluación más realista que mezclar filas al azar.
- **Importancia por permutación:** abre la "caja negra" y dice qué variables pesan.

**Ejercicio final:** agregue la **hora del día** como variable (como `sin` y `cos` de la hora, para que 23 h y 0 h queden cerca) y repita la comparación entre regresión lineal y red neuronal. ¿Mejora más la red o la lineal?